In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp
import json

# Load model & labels
model = tf.keras.models.load_model("saved_models/landmark_model.h5")
with open("saved_models/landmark_labels.json", "r") as f:
    class_names = json.load(f)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        lm = results.multi_hand_landmarks[0]
        coords = []
        for lm_point in lm.landmark:
            coords.extend([lm_point.x, lm_point.y, lm_point.z])
        coords = np.array(coords).reshape(1, -1)  # shape (1, 63)
        pred = model.predict(coords)[0]
        idx = np.argmax(pred)
        label = class_names[idx]
        confidence = pred[idx]

        cv2.putText(frame, f"{label} {confidence*100:.1f}%", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    else:
        # No hand detected
        cv2.putText(frame, "No Hand", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("Sign Translator", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()  this is teh code for tranlator .py

In [1]:
# scripts/train_model.py
import os, json, argparse
import tensorflow as tf

from tensorflow.keras import layers, models, callbacks

def build_model(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)
    # built-in augmentation (optional but useful)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomRotation(0.1)(x)

    x = layers.Conv2D(32, 3, activation="relu")(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def main(args):
    data_dir = os.path.join("dataset", "processed")
    train_dir = os.path.join(data_dir, "train")
    val_dir   = os.path.join(data_dir, "test")  # using test as validation here

    img_size = args.img_size
    batch = args.batch
    seed = 123

    train_ds = tf.keras.preprocessing.image_dataset_from_directory(train_dir,
                                                                  image_size=(img_size, img_size),
                                                                  batch_size=batch,
                                                                  seed=seed)
    val_ds = tf.keras.preprocessing.image_dataset_from_directory(val_dir,
                                                                image_size=(img_size, img_size),
                                                                batch_size=batch,
                                                                shuffle=False)

    class_names = train_ds.class_names
    print("Classes:", class_names)
    num_classes = len(class_names)

    # cache & prefetch
    train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

    model = build_model((img_size, img_size, 3), num_classes)
    model.summary()

    os.makedirs("saved_models", exist_ok=True)
    checkpoint = callbacks.ModelCheckpoint("saved_models/best_model.h5", save_best_only=True, monitor="val_accuracy", mode="max")
    early = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)

    hist = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs, callbacks=[checkpoint, early])
    # final save & labels
    model.save("saved_models/final_model.h5")
    with open("saved_models/labels.json", "w") as f:
        json.dump(class_names, f)

    # save history
    os.makedirs("outputs", exist_ok=True)
    with open("outputs/history.json", "w") as f:
        json.dump({k: [float(x) for x in v] for k,v in hist.history.items()}, f)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--img_size", type=int, default=128)
    p.add_argument("--batch", type=int, default=32)
    p.add_argument("--epochs", type=int, default=25)
    args = p.parse_args()
    main(args)


KeyboardInterrupt: 

In [ ]:
import os
import argparse
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def process_image(in_path, out_path, size):
    img = Image.open(in_path).convert("RGB")
    img = ImageOps.fit(img, (size, size), Image.Resampling.LANCZOS)
    img.save(out_path, quality=90)

def augment_and_save(img_path, out_dir, base_name, size):
    img = Image.open(img_path).convert("RGB")
    img = ImageOps.fit(img, (size, size), Image.Resampling.LANCZOS)
    # original
    img.save(os.path.join(out_dir, f"{base_name}.jpg"), quality=90)
    # flipped
    img.transpose(Image.FLIP_LEFT_RIGHT).save(os.path.join(out_dir, f"{base_name}_f.jpg"), quality=90)
    # small rotations
    for i, angle in enumerate([15, -15]):
        img.rotate(angle).save(os.path.join(out_dir, f"{base_name}_r{i}.jpg"), quality=90)

def main(args):
    src = os.path.join("dataset", "raw")
    dst = os.path.join("dataset", "processed")
    train_dir = os.path.join(dst, "train")
    test_dir = os.path.join(dst, "test")

    ensure_dir(train_dir)
    ensure_dir(test_dir)

    classes = [d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d))]
    print("Found classes:", classes)

    for cls in classes:
        cls_src = os.path.join(src, cls)
        images = [os.path.join(cls_src, f) for f in os.listdir(cls_src)
                  if f.lower().endswith((".png", ".jpg", ".jpeg"))]

        if not images:
            continue

        train_list, test_list = train_test_split(
            images, test_size=args.test_size, random_state=42, shuffle=True
        )

        ctrain = os.path.join(train_dir, cls)
        ctest = os.path.join(test_dir, cls)
        ensure_dir(ctrain)
        ensure_dir(ctest)

        for i, img_path in enumerate(train_list):
            base = f"{i}_{os.path.splitext(os.path.basename(img_path))[0]}"
            if args.augment:
                augment_and_save(img_path, ctrain, base, args.img_size)
            else:
                process_image(img_path, os.path.join(ctrain, f"{base}.jpg"), args.img_size)

        for i, img_path in enumerate(test_list):
            base = f"{i}_{os.path.splitext(os.path.basename(img_path))[0]}"
            process_image(img_path, os.path.join(ctest, f"{base}.jpg"), args.img_size)

    print("Preprocessing done. Check dataset/processed/")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--img-size", type=int, default=128)
    parser.add_argument("--test-size", type=float, default=0.2)
    parser.add_argument("--augment", action="store_true", help="create flips/rotations for training")
    args = parser.parse_args()
    main(args)


In [ ]:
cv2.imshow("ROI", roi)  # roi is the cropped region for prediction
cv2.waitKey(1)          # 1 ms wait to update the window
